# C01–M1–L1 — Master Prompt Engineering and LangChain PromptTemplates (Colab edition · Gemini stack)

Converted from the IBM Skills Network lab · [original notebook/files](https://github.com/shayan127/IBM-RAG-and-Agentic-AI/blob/master/01-Develop-Generative-AI-Applications-Get-Started/resources/In-Context%20Learning%20and%20Prompt%20Templates-v3-GenAIcourse.ipynb)

**Before you run:** add your free `GOOGLE_API_KEY` ([Google AI Studio](https://aistudio.google.com/apikey)) to Colab 🔑 Secrets and enable notebook access. No GPU needed — a CPU runtime is enough.

**What changed vs. the IBM version**
- IBM install + kernel-restart cells removed → Cell 1 = minimal Gemini install, Cell 2 = universal Gemini init (`llm`, `embeddings`, `llm_model`).
- watsonx Granite / Llama → `gemini-3.8-flash` via `ChatGoogleGenerativeAI`; `llm_model(prompt, params)` is now the 1-line alias (params accepted, ignored).
- `WatsonxLLM(...)` + `GenParams` cells (LCEL section, Exercise 5) → `ChatGoogleGenerativeAI(...)`; `GenParams().get_example_values()` → Gemini generation settings.
- `format_prompt` + `RunnableLambda(format_prompt)` removed everywhere — chains are plain LCEL `prompt | llm | StrOutputParser()`.
- Legacy `from langchain.chains import LLMChain` import dropped (unused; LangChain 1.x moved it to `langchain-classic`).
- API-key / Skills Network disclaimer cells removed; Colab exercises at the end.

Everything else — explanations, code, exercises — is the original lab.

In [ ]:
# Minimal fast install (~5-10 seconds)
!pip install -q -U langchain-google-genai langchain-core langchain-chroma google-genai

### 🔑 Gemini initialization
Run once. Creates `llm` (`gemini-3.8-flash`), `embeddings` (`models/gemini-embedding-2`) and the 1-line `llm_model()` alias used by older lab cells. Needs the free `GOOGLE_API_KEY` from [Google AI Studio](https://aistudio.google.com/apikey) in Colab 🔑 Secrets (notebook access on).

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# Load API key securely from Colab Secrets (with graceful fallback)
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    pass

if not api_key:
    api_key = os.environ.get("GOOGLE_API_KEY")

if not api_key:
    import getpass
    print("🔑 Tip: Add 'GOOGLE_API_KEY' in the Colab 🔑 Secrets tab and toggle 'Notebook access' ON to load automatically.")
    api_key = getpass.getpass("Enter your Google AI Studio API key: ")

os.environ["GOOGLE_API_KEY"] = api_key

# Initialize LLM and Embeddings directly (no helper functions or custom shims)
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.2)
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

# Backward-compatibility alias for older lab cells calling llm_model()
# (.text, not .content: Gemini 3.x can return content as a list of blocks; .text is always a str)
llm_model = lambda prompt, params=None: llm.invoke(prompt).text

print("✅ Google Gemini 3.8 Flash & Gemini Embedding 2 initialized successfully.")

# Master Prompt Engineering and LangChain PromptTemplates


Estimated time needed: **45** minutes


## Overview


You're stepping into the world of prompt engineering, where each command you craft has the power to guide intelligent LLM systems toward specific outcomes. In this tutorial, you will explore the foundational aspects of prompt engineering, dive into advanced in-context learning techniques such as few-shot and self-consistent learning, and learn how to effectively use tools like LangChain to create prompt templates.

You'll start by learning the basics—how to formulate prompts that communicate effectively with AI. From there, you'll explore how LangChain prompt templates can simplify and enhance this process, making it more structured and efficient.

As you progress, you'll learn to apply these skills in practical scenarios, creating sophisticated applications like QA bots and text summarization tools. By using LangChain prompt templates, you'll see firsthand how structured prompting can streamline the development of these applications, transforming complex requirements into clear, concise tasks for AI.


<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/ai8G4tOU4mksEYfv5wsghA/prompt%20engineering.png" width="50%" alt="indexing"/>


By the end of this tutorial, you'll not only master various prompt engineering techniques but also gain hands-on experience applying these techniques to real-world problems, ensuring you're well-prepared to harness the full potential of AI in diverse settings.


## __Table of Contents__

<ol>
    <li><a href="#Objectives">Objectives</a></li>
    <li>
        <a href="#Setup">Setup</a>
        <ol>
            <li><a href="#Install-required-libraries">Install required libraries</a></li>
            <li><a href="#Import-required-libraries">Import required libraries</a></li>
            <li><a href="#Set-up-the-LLM">Set up the LLM</a></li>
        </ol>
    </li>
    <li>
        <a href="#Prompt-engineering">Prompt engineering</a>
        <ol>
            <li>
                <a href="#Basic-prompt">Basic prompt</a>
                <ol>
                    <li><a href="#Exercise-1">Exercise 1</a></li>
                </ol>
            </li>
            <li>
                <a href="#Zero-shot-prompt">Zero-shot prompt</a>
                <ol>
                    <li><a href="#Exercise-2">Exercise 2</a></li>
                </ol>
            </li>
            <li>
                <a href="#One-shot-prompt">One-shot prompt</a>
                <ol>
                    <li><a href="#Exercise-3">Exercise 3</a></li>
                </ol>
            </li>
            <li><a href="#Few-shot-prompt">Few-shot prompt</a></li>
            <li>
                <a href="#Chain-of-thought-(CoT)-prompt">Chain-of-thought (CoT) prompt</a>
                <ol>
                    <li><a href="#Exercise-4">Exercise 4</a></li>
                </ol>
            </li>
            <li><a href="#Self-consistency">Self-consistency</a></li>
        </ol>
    </li>
    <li>
        <a href="#Applications-of-prompting-in-different-use-cases">Applications of prompting in different use cases</a>
        <ol>
            <li><a href="#Introduction-to-LangChain">Introduction to LangChain</a></li>
            <li><a href="#Prompt-template">Prompt template</a></li>
            <li><a href="#Text-summarization">Text summarization</a></li>
            <li><a href="#Question-answering">Question answering</a></li>
            <li><a href="#Text-classification">Text classification</a></li>
            <li><a href="#Code-generation">Code generation</a></li>
            <li><a href="#Role-playing">Role playing</a></li>
            <li><a href="#Exercise-5">Exercise 5</a></li>
        </ol>
    </li>
    <li><a href="#Conclusion">Conclusion</a></li>
    <li><a href="#Authors">Authors</a></li>
</ol>


## Objectives

After completing this lab, you will be able to:

- **Understand the basics of prompt engineering**: Gain a solid foundation in how to effectively communicate with LLM using prompts, setting the stage for more advanced techniques.

- **Master advanced prompt techniques**: Learn and apply advanced prompt engineering methods such as few-shot and self-consistent learning to optimize the LLM's response.

- **Utilize LangChain prompt templates**: Become proficient in using LangChain's prompt template to structure and optimize your interactions with LLMs.

- **Develop practical LLM agents**: Acquire the skills to create and implement agents such as QA bots and text summarization tools using the LangChain prompt template, translating theoretical knowledge into practical solutions.


----


## Setup


### Import required libraries


_It is recommended that you import all required libraries in one place (here):_


In [ ]:
# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableSequence
from langchain_core.messages import HumanMessage, SystemMessage

### Set up the LLM


In this section, you will build an LLM using IBM watsonx.ai. The following code initializes a Granite model on IBM's watsonx.ai platform and wraps it into a function that allows for repeat use.

> **Colab (Gemini):** instead of watsonx Granite, `llm_model()` now sends the prompt to `gemini-3.8-flash` (initialized in Cell 2).

Some key parameters are explained here:
- `model_id` specifies which model you want to use. There are various model options available; refer to the [Foundation Models](https://ibm.github.io/watsonx-ai-python-sdk/foundation_models.html) documentation for more options. In this tutorial, you'll use the `ibm/granite-4-h-small` model.
- `parameters` define the model's configuration. Set five commonly used parameters for this tutorial. To explore additional commonly used parameters, you can run the code `GenParams().get_example_values()`. If no custom parameters are passed to the function, the model will use `default_params`.
- `credentials` and `project_id` are required to successfully run LLMs from watsonx.ai. (Leave `credentials` and `project_id` as they are so you don't need to create your own keys to run models.) This ensures you can run the model inside this lab environment. However, if you want to run the model locally, refer to this [tutorial](https://medium.com/the-power-of-ai/ibm-watsonx-ai-the-interface-and-api-e8e1c7227358) for creating your own keys.
- `WatsonxLLM()` is used to create an instance of the LLM.

> **Colab (Gemini):** Gemini's equivalents are `temperature`, `top_p`, `top_k`, `max_output_tokens` and `stop`, set on `ChatGoogleGenerativeAI(...)`. No credentials or `project_id` are needed — only `GOOGLE_API_KEY`.

In [ ]:
# Colab (Gemini): the watsonx Granite helper is replaced by the 1-line alias from Cell 2.
# `params` is still accepted so the lab's calls work unchanged; to change sampling, rebuild `llm`, e.g.
#   llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.8)
llm_model = lambda prompt, params=None: llm.invoke(prompt).text

Let's run the following code to see some other commonly used parameters and their default values.


In [ ]:
# Colab (Gemini): GenParams().get_example_values() → the generation settings of the Gemini chat model
{k: getattr(llm, k, None) for k in ["model", "temperature", "top_p", "top_k", "max_output_tokens", "stop"]}

## Prompt engineering

[Prompt engineering](https://www.ibm.com/think/topics/prompt-engineering?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-In-Context+Learning+and+Prompt+Templates-v3-GenAIcourse_1741386184) is the art and science of crafting effective inputs for large language models to generate desired outputs. As language models have evolved in capability and size, so too has the importance of how we communicate with them. Prompt engineering involves strategically designing text prompts that guide an AI model's responses toward specific goals, formats, or reasoning patterns.

**[In-context learning](https://research.ibm.com/blog/demystifying-in-context-learning-in-large-language-model?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-In-Context+Learning+and+Prompt+Templates-v3-GenAIcourse_1741386184)** represents one of the most fascinating capabilities of modern large language models. It is a model's ability to "learn" from examples provided directly within the prompt itself, without any updates to its underlying parameters or weights. This capability allows models to adapt to new tasks or domains simply by demonstrating what success looks like through examples.

By combining the principles of prompt engineering with the power of in-context learning, developers can guide language models to perform a remarkably diverse range of tasks with unprecedented flexibility and efficiency. Let's now explore these methods in detail.


### Basic prompt

A **basic prompt** is the simplest form of prompting, where you provide a short text or phrase to the model without any special formatting or instructions. The model generates a continuation based on patterns it has learned during training. Basic prompts are useful for exploring the model's capabilities and understanding how it naturally responds to minimal input.


In this example, let's introduce a basic prompt that uses specific parameters to guide the language model's response. You'll then define a simple prompt and retrieve the model's response.

The prompt used is "The wind is." Let the model generate itself.


In [ ]:
params = {
    "max_new_tokens": 128,
    "min_new_tokens": 10,
    "temperature": 0.5,
    "top_p": 0.2,
    "top_k": 1
}

prompt = "The wind is "

# Getting a reponse from the model with the provided prompt and new parameters
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

As you can see from the response, the model continues generating content based on the initial prompt, "The wind is." You might notice that the response appears truncated or incomplete. This is because you have set the `max_new_tokens,` which restricts the number of tokens the model can generate.

Try adjusting the parameters and observe how the response changes.


### Exercise 1
Experiment with different basic prompts by changing the input phrase. Try these prompts and compare the different responses:
1. "The future of artificial intelligence is"
2. "Once upon a time in a distant galaxy"
3. "The benefits of sustainable energy include"


In [ ]:
## Your code here


<details>
    <summary>Click here for hints</summary>

```python
params = {
    "max_new_tokens": 128, # Try 256 or 512 for more detailed answers
    "min_new_tokens": 10, # Increase to 25-50 if you want more substantial answers
    "temperature": 0.5, # Controls randomness in generation (0.0-1.0)
                       # Lower (0.1-0.3): More focused, consistent, factual responses
                      # Higher (0.7-1.0): More creative, diverse, unpredictable outputs
    "top_p": 0.2, # Nucleus sampling - considers only highest probability tokens
                       # Lower values (0.1-0.3): More conservative, focused text
                       # Higher values (0.7-0.9): More diverse vocabulary and ideas
    "top_k": 1 # Limits token selection to top k most likely tokens
                       # 1 = greedy decoding (always picks most likely token)
                       # Try 40-50 for more varied outputs
}

# Compare responses to different prompts
prompts = [
    "The future of artificial intelligence is",
    "Once upon a time in a distant galaxy",
    "The benefits of sustainable energy include"
]

for prompt in prompts:
    response = llm_model(prompt, params)
    print(f"prompt: {prompt}\n")
    print(f"response : {response}\n")
```
</details>


> 💡 **✅ Answer cells** (code hidden in Colab) follow every exercise. Try the exercise in the cell above first. Then click **▸ Show code** on the ✅ cell to read the answer, or tick **run_answer** and run it. Unticked answer cells stop *Run all* on purpose, so you meet each exercise before its answer — use *Runtime → Run after* to continue.

In [ ]:
# @title ✅ Answer — Exercise 1 — Basic prompts
run_answer = False  # @param {type:"boolean"}
# Try the exercise yourself first. Then tick run_answer (Colab) or set it to True, and run this cell.
# To read the code: click ▸ Show code (or double-click the cell title).
if not run_answer:
    class AnswerSkipped(Exception):
        def _render_traceback_(self): return ["⏸ Answer not run — tick run_answer above after you have tried the exercise."]
    raise AnswerSkipped()

# Exercise 1 — basic prompts
params = {"max_new_tokens": 128, "min_new_tokens": 10, "temperature": 0.5, "top_p": 0.2, "top_k": 1}

prompts = [
    "The future of artificial intelligence is",
    "Once upon a time in a distant galaxy",
    "The benefits of sustainable energy include",
]

for prompt in prompts:
    response = llm_model(prompt, params)   # params are accepted but ignored by the Gemini alias
    print(f"prompt: {prompt}\n")
    print(f"response : {response}\n")

### Zero-shot prompt

[**Zero-shot prompting**](https://www.ibm.com/think/topics/zero-shot-prompting?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-In-Context+Learning+and+Prompt+Templates-v3-GenAIcourse_1741386184) is a technique where the model performs a task without any examples or prior specific training on that task. This approach tests the model's ability to understand instructions and apply its knowledge to a new context without demonstration. Zero-shot prompts typically include clear instructions about what the model should do, allowing it to leverage its pre-trained knowledge effectively.


---
Here is an example of a zero-shot prompt:

Zero-shot learning is crucial for testing a model's ability to apply its pre-trained knowledge to new, unseen tasks without additional training. This capability is valuable for gauging the model's generalization skills.

In this example, let's demonstrate a zero-shot learning scenario using a prompt that asks the model to classify a statement without any prior specific training on similar tasks. The prompt requests the model to assess the truthfulness of the statement: "The Eiffel Tower is located in Berlin." After defining the prompt, you'll execute it with default parameters and print the response.

This approach helps you understand how well the model can handle direct questions based on its underlying knowledge and reasoning abilities.


Try running the prompt to see the model's capacity to correctly analyze and respond to factual inaccuracies.


In [ ]:
prompt = """Classify the following statement as true or false: 
            'The Eiffel Tower is located in Berlin.'

            Answer:
"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

The model responds with the 'False' answer, which is correct. It also gives the reason for the answer.


### Exercise 2
Create zero-shot prompts for the following tasks:
1. Write a prompt that asks the model to classify a movie review as positive or negative.
2. Create a prompt that instructs the model to summarize a paragraph about climate change.
3. Design a prompt that asks the model to translate an English phrase to Spanish without showing any examples.


In [ ]:
## Starter code: provide your solutions in the TODO parts

# 1. Prompt for Movie Review Classification
movie_review_prompt = ## TODO

# 2. Prompt for Climate Change Paragraph Summarization
climate_change_prompt = ## TODO

# 3. Prompt for English to Spanish Translation
translation_prompt = ## TODO

responses = {}
responses["movie_review"] = ## TODO
responses["climate_change"] = ## TODO
responses["translation"] = ## TODO

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

<details>
    <summary>Click here for hints</summary>

```python
# 1. Prompt for Movie Review Classification
movie_review_prompt = """
Classify the following movie review as either 'positive' or 'negative'.

Review: "I was extremely disappointed by this film. The plot was predictable, the acting was wooden, and the special effects looked cheap. I can't recommend this to anyone."

Classification:
"""

# 2. Prompt for Climate Change Paragraph Summarization
climate_change_prompt = """
Summarize the following paragraph about climate change in no more than two sentences.

Paragraph: "Climate change refers to long-term shifts in temperatures and weather patterns. These shifts may be natural, but since the 1800s, human activities have been the main driver of climate change, primarily due to the burning of fossil fuels like coal, oil and gas, which produces heat-trapping gases. The consequences of climate change include more frequent and severe droughts, storms, and heat waves, rising sea levels, melting glaciers, and warming oceans which can directly impact biodiversity, agriculture, and human health."

Summary:
"""

# 3. Prompt for English to Spanish Translation
translation_prompt = """
Translate the following English phrase into Spanish.

English: "I would like to order a coffee with milk and two sugars, please."

Spanish:
"""

responses = {}
responses["movie_review"] = llm_model(movie_review_prompt)
responses["climate_change"] = llm_model(climate_change_prompt)
responses["translation"] = llm_model(translation_prompt)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()
```
</details>


In [ ]:
# @title ✅ Answer — Exercise 2 — Zero-shot prompts
run_answer = False  # @param {type:"boolean"}
# Try the exercise yourself first. Then tick run_answer (Colab) or set it to True, and run this cell.
# To read the code: click ▸ Show code (or double-click the cell title).
if not run_answer:
    class AnswerSkipped(Exception):
        def _render_traceback_(self): return ["⏸ Answer not run — tick run_answer above after you have tried the exercise."]
    raise AnswerSkipped()

# Exercise 2 — zero-shot prompts
# 1. Movie review classification
movie_review_prompt = """Classify the following movie review as either 'positive' or 'negative'.
Answer with one word only.

Review: "I was extremely disappointed by this film. The plot was predictable, the acting was wooden,
and the special effects looked cheap. I wouldn't recommend it to anyone."

Classification:"""

# 2. Climate-change paragraph summarization
climate_change_prompt = """Summarize the following paragraph in one or two sentences.

Paragraph: "Climate change refers to long-term shifts in temperatures and weather patterns. Since the
1800s, human activities have been the main driver of climate change, primarily due to burning fossil
fuels like coal, oil and gas, which produces heat-trapping gases. The consequences include rising sea
levels, more frequent extreme weather events, and threats to biodiversity and food security."

Summary:"""

# 3. English → Spanish translation (no examples = zero-shot)
translation_prompt = """Translate the following English phrase into Spanish. Return only the translation.

English: "Where is the nearest train station?"

Spanish:"""

responses = {}
responses["movie_review"] = llm_model(movie_review_prompt, params)
responses["climate_change"] = llm_model(climate_change_prompt, params)
responses["translation"] = llm_model(translation_prompt, params)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

### One-shot prompt

[**One-shot prompting**](https://www.ibm.com/think/topics/one-shot-prompting?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-In-Context+Learning+and+Prompt+Templates-v3-GenAIcourse_1741386184) provides the model with a **single** example of the task before asking it to perform a similar task. This technique gives the model a pattern to follow, improving its understanding of the desired output format and style. One-shot learning is particularly useful when you want to guide the model's response without extensive examples.


Here is a one-shot learning example where the model is given a single example to help guide its translation from English to French.

The prompt provides a sample translation pairing, "How is the weather today?" translated to "Comment est le temps aujourd'hui?" This example serves as a guide for the model to understand the task context and desired format. The model is then tasked with translating a new sentence, "Where is the nearest supermarket?" without further guidance.


In [ ]:
params = {
    "max_new_tokens": 20,
    "temperature": 0.1,
}

prompt = """Here is an example of translating a sentence from English to French:

            English: “How is the weather today?”
            French: “Comment est le temps aujourd'hui?”
            
            Now, translate the following sentence from English to French:
            
            English: “Where is the nearest supermarket?”
            
"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

The model's response shows how it applies the structure and context provided by the initial example to translate the new sentence.


Consider experimenting with different sentences or adjusting the parameters to see how these changes impact the model's translations.


### Exercise 3
Develop one-shot prompts for these scenarios:
1. Create a prompt with one example of a formal email, then ask the model to write another formal email on a different topic.
2. Provide one example of converting a technical concept into a simple explanation, then ask the model to explain a different concept.
3. Give one example of extracting keywords from a sentence, then ask the model to extract keywords from a new sentence.


In [ ]:
## Starter code: provide your solutions in the TODO parts

# 1. One-shot prompt for formal email writing
formal_email_prompt = ## TODO

# 2. One-shot prompt for simplifying technical concepts
technical_concept_prompt = ## TODO

# 3. One-shot prompt for keyword extraction
keyword_extraction_prompt = ## TODO

responses = {}
responses["formal_email"] = ## TODO
responses["technical_concept"] = ## TODO
responses["keyword_extraction"] = ## TODO

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

<details>
    <summary>Click here for hints</summary>

```python
# 1. One-shot prompt for formal email writing
formal_email_prompt = """
Here is an example of a formal email requesting information:

Subject: Inquiry Regarding Product Specifications for Model XYZ-100

Dear Customer Support Team,

I hope this email finds you well. I am writing to request detailed specifications for your product Model XYZ-100. Specifically, I am interested in learning about its dimensions, power requirements, and compatibility with third-party accessories.

Could you please provide this information at your earliest convenience? Additionally, I would appreciate any available documentation or user manuals that you could share.

Thank you for your assistance in this matter.

Sincerely,
John Smith

---

Now, please write a formal email to a university admissions office requesting information about their application deadline and required documents for the Master's program in Computer Science:

"""

# 2. One-shot prompt for simplifying technical concepts
technical_concept_prompt = """
Here is an example of explaining a technical concept in simple terms:

Technical Concept: Blockchain
Simple Explanation: A blockchain is like a digital notebook that many people have copies of. When someone writes a new entry in this notebook, everyone's copy gets updated. Once something is written, it can't be erased or changed, and everyone can see who wrote what. This makes it useful for recording important information that needs to be secure and trusted by everyone.

---

Now, please explain the following technical concept in simple terms:

Technical Concept: Machine Learning
Simple Explanation:
"""

# 3. One-shot prompt for keyword extraction
keyword_extraction_prompt = """
Here is an example of extracting keywords from a sentence:

Sentence: "Cloud computing offers businesses flexibility, scalability, and cost-efficiency for their IT infrastructure needs."
Keywords: cloud computing, flexibility, scalability, cost-efficiency, IT infrastructure

---

Now, please extract the main keywords from the following sentence:

Sentence: "Sustainable agriculture practices focus on biodiversity, soil health, water conservation, and reducing chemical inputs."
Keywords:
"""

responses = {}
responses["formal_email"] = llm_model(formal_email_prompt)
responses["technical_concept"] = llm_model(technical_concept_prompt)
responses["keyword_extraction"] = llm_model(keyword_extraction_prompt)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()
```
</details>


In [ ]:
# @title ✅ Answer — Exercise 3 — One-shot prompts
run_answer = False  # @param {type:"boolean"}
# Try the exercise yourself first. Then tick run_answer (Colab) or set it to True, and run this cell.
# To read the code: click ▸ Show code (or double-click the cell title).
if not run_answer:
    class AnswerSkipped(Exception):
        def _render_traceback_(self): return ["⏸ Answer not run — tick run_answer above after you have tried the exercise."]
    raise AnswerSkipped()

# Exercise 3 — one-shot prompts
# 1. Formal email: one example, then a new topic
formal_email_prompt = """Here is an example of a formal email requesting information:

Subject: Inquiry Regarding Product Specifications for Model XYZ-100

Dear Customer Support Team,

I hope this message finds you well. I am writing to request detailed specifications for your
Model XYZ-100, specifically its power consumption and warranty terms.

Thank you for your assistance. I look forward to your reply.

Sincerely,
John Smith

Now, write a formal email in the same style to request a meeting with a university professor
to discuss research opportunities in machine learning."""

# 2. Technical concept → simple explanation
technical_concept_prompt = """Here is an example of explaining a technical concept in simple terms:

Technical concept: Cloud computing
Simple explanation: Cloud computing is like renting a computer that lives somewhere else. Instead of
keeping all your files and programs on your own computer, you use the internet to reach a powerful
computer owned by a company, and you pay only for what you use.

Now explain the following technical concept in the same simple way:

Technical concept: Machine learning
Simple explanation:"""

# 3. Keyword extraction
keyword_extraction_prompt = """Here is an example of extracting keywords from a sentence:

Sentence: "Artificial intelligence is transforming healthcare by improving diagnostic accuracy and treatment planning."
Keywords: artificial intelligence, healthcare, diagnostic accuracy, treatment planning

Now extract the keywords from this sentence in the same format:

Sentence: "Renewable energy sources like solar and wind power are reducing global carbon emissions."
Keywords:"""

responses = {}
responses["formal_email"] = llm_model(formal_email_prompt, params)
responses["technical_concept"] = llm_model(technical_concept_prompt, params)
responses["keyword_extraction"] = llm_model(keyword_extraction_prompt, params)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

### Few-shot prompt

[**Few-shot prompting**](https://www.ibm.com/think/topics/few-shot-prompting?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-In-Context+Learning+and+Prompt+Templates-v3-GenAIcourse_1741386184) extends the one-shot approach by providing multiple examples (typically 2-5) before asking the model to perform the task. These examples establish a clearer pattern and context, helping the model better understand the expected output format, style, and reasoning. This technique is particularly effective for complex tasks where a single example might not convey all the nuances.


Here is an example of few-shot learning by classifying emotions from text statements. 

Let's provide the model with three examples, each labeled with an appropriate emotion—joy, frustration, and sadness—to establish a pattern or guideline on how to categorize emotions in statements.

After presenting these examples, let's challenge the model with a new statement: "That movie was so scary I had to cover my eyes." The task for the model is to classify the emotion expressed in this new statement based on the learning from the provided examples. 


In [ ]:
#parameters: Set `max_new_tokens` to 10, which constrains the model to generate brief responses

params = {
    "max_new_tokens": 10,
}

prompt = """Here are few examples of classifying emotions in statements:

            Statement: 'I just won my first marathon!'
            Emotion: Joy
            
            Statement: 'I can't believe I lost my keys again.'
            Emotion: Frustration
            
            Statement: 'My best friend is moving to another country.'
            Emotion: Sadness
            
            Now, classify the emotion in the following statement:
            Statement: 'That movie was so scary I had to cover my eyes.’
            

"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

The parameters are set with `max_new_tokens` to 10, which constrains the model to generate brief responses, focusing on the essential output without elaboration.


The model's response demonstrates its ability to use the provided few examples to understand and classify the emotion of the new statement effectively following the same pattern in examples.


### Chain-of-thought (CoT) prompt

[**Chain-of-thought (CoT) prompting**](https://www.ibm.com/think/topics/chain-of-thoughts?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-In-Context+Learning+and+Prompt+Templates-v3-GenAIcourse_1741386184) encourages the model to break down complex problems into step-by-step reasoning before arriving at a final answer. By explicitly showing or requesting intermediate steps, this technique improves the model's problem-solving abilities and reduces errors in tasks requiring multi-step reasoning. CoT is particularly effective for mathematical problems, logical reasoning, and complex decision-making tasks.


Here is an example of the CoT prompting technique, designed to guide the model through a sequence of reasoning steps to solve a problem. In this example, the problem is a simple arithmetic question: “A store had 22 apples. They sold 15 apples today and received a new delivery of 8 apples. How many apples are there now?”

The CoT technique involves structuring the prompt by instructing the model to “Break down each step of your calculation.” This encourages the model to include explicit reasoning steps, mimicking human-like problem-solving processes.


In [ ]:
params = {
    "max_new_tokens": 512,
    "temperature": 0.5,
}

prompt = """Consider the problem: 'A store had 22 apples. They sold 15 apples today and got a new delivery of 8 apples. 
            How many apples are there now?’

            Break down each step of your calculation

"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

From the response of the model, you can see the prompt directs the model to:

1. Add the initial number of apples to the apples received in the new delivery.
2. Subtract the number of apples sold from the sum obtained in the first step.


By breaking down the problem into specific steps, the model is better able to understand the sequence of operations required to arrive at the correct answer.


### Exercise 4
Create CoT prompts for these scenarios:
1. Write a prompt that asks the model to think through whether a student should study tonight or go to a movie with friends, considering their upcoming test in two days.
2. Write a prompt that instructs the model to explain the step-by-step process of making a peanut butter and jelly sandwich.


In [ ]:
## Starter code: provide your solutions in the TODO parts

# 1. Prompt for decision-making process
decision_making_prompt = ## TODO

# 2. Prompt for explaining a process
sandwich_making_prompt = ## TODO

responses = {}
responses["decision_making"] = ## TODO
responses["sandwich_making"] = ## TODO

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

<details>
    <summary>Click here for hints</summary>

```python
# 1. Prompt for decision-making process
decision_making_prompt = """
Consider this situation: A student is trying to decide whether to study tonight or go to a movie with friends. They have a test in two days.

Think through this decision step-by-step, considering the pros and cons of each option, and what factors might be most important in making this choice.
"""

# 2. Prompt for explaining a process
sandwich_making_prompt = """
Explain how to make a peanut butter and jelly sandwich.

Break down each step of the process in detail, from gathering ingredients to finishing the sandwich.
"""

responses = {}
responses["decision_making"] = llm_model(decision_making_prompt)
responses["sandwich_making"] = llm_model(sandwich_making_prompt)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()
```
</details>


In [ ]:
# @title ✅ Answer — Exercise 4 — Chain-of-thought prompts
run_answer = False  # @param {type:"boolean"}
# Try the exercise yourself first. Then tick run_answer (Colab) or set it to True, and run this cell.
# To read the code: click ▸ Show code (or double-click the cell title).
if not run_answer:
    class AnswerSkipped(Exception):
        def _render_traceback_(self): return ["⏸ Answer not run — tick run_answer above after you have tried the exercise."]
    raise AnswerSkipped()

# Exercise 4 — chain-of-thought prompts
# 1. Decision making
decision_making_prompt = """Consider this situation: A student is trying to decide whether to study tonight
or go to a movie with friends. They have a test in two days.

Think through this decision step by step:
1. List the factors the student should consider (how prepared they are, how hard the test is,
   how much study time is left, the value of rest and social time).
2. Weigh the pros and cons of each option.
3. Consider a compromise (for example, study first, then go to a later show).
4. Give a final recommendation and explain why."""

# 2. Step-by-step process
sandwich_making_prompt = """Explain the process of making a peanut butter and jelly sandwich.
Think step by step: first list the ingredients and tools you need, then describe every step in
order as a numbered list, and briefly say why each step matters."""

responses = {}
responses["decision_making"] = llm_model(decision_making_prompt, params)
responses["sandwich_making"] = llm_model(sandwich_making_prompt, params)

for prompt_type, response in responses.items():
    print(f"=== {prompt_type.upper()} RESPONSE ===")
    print(response)
    print()

### Self-consistency

[**Self-consistency**](https://www.promptingguide.ai/techniques/consistency) is an advanced technique in which the model generates multiple independent solutions or answers to the same problem, then evaluates these different approaches to determine the most consistent or reliable result. This method enhances accuracy by leveraging the model's ability to approach problems from different angles and identify the most robust solution through comparison and verification.


This example demonstrates the self-consistency technique by reasoning through multiple calculations for a single problem. The problem posed is: “When I was 6, my sister was half my age. Now I am 70, what age is my sister?”

The prompt instructs, “Provide three independent calculations and explanations, then determine the most consistent result.” This encourages the model to engage in critical thinking and consistency checking, both of which are vital for complex decision-making processes.


In [ ]:
params = {
    "max_new_tokens": 512,
}

prompt = """When I was 6, my sister was half of my age. Now I am 70, what age is my sister?

            Provide three independent calculations and explanations, then determine the most consistent result.

"""
response = llm_model(prompt, params)
print(f"prompt: {prompt}\n")
print(f"response : {response}\n")

The model's response demonstrates three different calculations and explanations, each using a distinct logical approach to determine the sister's age.

Self-consistency can help identify the most accurate and reliable answer in scenarios where multiple plausible solutions exist.


## Applications of prompting in different use cases


In this section, we'll demonstrate how to leverage LangChain's prompt templates to build practical applications with consistent, reproducible results. Each application follows a common pattern using the LCEL approach:

1. Define the content or problem to be addressed.
2. Create a template with variables for dynamic content.
3. Convert the template into a LangChain PromptTemplate.
4. Build a chain using the pipe operator `|` to connect:

    - Input variables
    - The prompt template
    - The LLM
    - An output parser


5. Invoke the chain with specific inputs to generate results.

This structured approach enables you to create reusable components for various NLP tasks while maintaining flexibility to adjust parameters and inputs. You'll see how this pattern applies across different use cases.


### Introduction to LangChain 

[LangChain](https://www.langchain.com/) is a powerful framework designed to simplify the development of applications powered by language models. Built to address the challenges of working with LLMs in practical settings, LangChain provides a standardized interface for connecting models with various data sources and application environments.

LangChain serves as an abstraction layer, making it easier to build complex LLM applications without handling the low-level details of model interaction. This framework has become a standard tool in the LLM ecosystem, supporting a wide range of use cases from chatbots to document analysis systems.

In this section, we'll focus on LangChain's prompt template capabilities, demonstrating how they can be used to create structured, reproducible interactions with language models across different application types."


### Prompt template


[Prompt templates](https://python.langchain.com/v0.2/docs/concepts/#prompt-templates) are a key concept in LangChain. They help translate user input and parameters into instructions for a language model. These templates can be used to guide a model's response, helping it understand the context and generate relevant and coherent language-based outputs.

A prompt template acts as a reusable structure for generating prompts with dynamic values. It allows you to define a consistent format while leaving placeholders for variables that change with each use case. This approach makes prompting more systematic and maintainable, especially when working with complex applications.

**Modern LangChain (as of 2025) offers two main approaches to working with templates:**

- The traditional `LLMChain` approach
- The newer LangChain Expression Language (LCEL) pattern using the pipe operator `|` for more flexible composition

LCEL has become the recommended pattern for building LangChain applications as it offers better composability, clearer visualization of data flow, and more flexibility when constructing complex chains.

**To use a prompt template with LCEL, you typically follow these steps:**

- Define your template with variables in curly braces `{}`
- Create a `PromptTemplate` instance
- Build a chain using the pipe operator `|` to connect components
- Invoke the chain with your input values

Let's initialize an LLM first, then demonstrate this approach. In this section, we will use the model `meta-llama/llama-3-3-70b-instruct` (Colab: `gemini-3.8-flash`):


In [ ]:
# Colab (Gemini): WatsonxLLM(model_id, url, project_id, params) → ChatGoogleGenerativeAI
model_id = "gemini-3.8-flash"   # was ibm/granite-4-h-small on watsonx.ai

llm = ChatGoogleGenerativeAI(
    model=model_id,
    temperature=0.5,          # GenParams.TEMPERATURE
    max_output_tokens=1024,   # GenParams.MAX_NEW_TOKENS (leave headroom: Gemini 3 also spends tokens on thinking)
)
llm

Use the `PromptTemplate` to create a template for a string-based prompt. In this template, you'll define two parameters: `adjective` and `content`. These parameters allow for the reuse of the prompt across different situations. For instance, to adapt the prompt to various contexts, simply pass the relevant values to these parameters.


In [ ]:
template = """Tell me a {adjective} joke about {content}.
"""
prompt = PromptTemplate.from_template(template)
prompt 

Now, let's take a look at how the prompt has been formatted.


In [ ]:
prompt.format(adjective="funny", content="chickens")

From the response, you can see that the prompt is formatted according to the specified context.


In LCEL a `PromptTemplate` is itself a *Runnable*: it accepts the dictionary of variables (like `{adjective}` and `{content}`) and fills the placeholders before the prompt is sent to the language model. So there is no need for a separate `format_prompt` helper or `RunnableLambda` — you pipe the dictionary straight into the prompt.

The following code builds a chain using the LCEL (LangChain Expression Language) pattern. This chain connects components using the pipe operator (`|`) to create a processing flow. The chain takes input variables, passes them through the prompt template, sends the formatted prompt to the LLM, and uses a string output parser to return the final response.


In [ ]:
# Create the chain with explicit formatting
joke_chain = (
    prompt
    | llm 
    | StrOutputParser()
)

# Run the chain
response = joke_chain.invoke({"adjective": "funny", "content": "chickens"})
print(response)

From the response, you can see the LLM came up with a funny joke about chickens.

To use this prompt in another context, simply replace the variables accordingly.


In [ ]:
response = joke_chain.invoke({"adjective": "sad", "content": "fish"})
print(response)

In the following sections, you will learn how to create agents capable of completing various tasks using prompt templates.


### Text summarization


Here is a text summarization agent designed to help summarize the content you provide to the LLM. The LCEL chain takes your content as input, processes it through the prompt template, sends it to the language model, and returns a concise summary.

You can store the content to be summarized in a variable, allowing for repeated use with different texts:


In [ ]:
content = """
    The rapid advancement of technology in the 21st century has transformed various industries, including healthcare, education, and transportation. 
    Innovations such as artificial intelligence, machine learning, and the Internet of Things have revolutionized how we approach everyday tasks and complex problems. 
    For instance, AI-powered diagnostic tools are improving the accuracy and speed of medical diagnoses, while smart transportation systems are making cities more efficient and reducing traffic congestion. 
    Moreover, online learning platforms are making education more accessible to people around the world, breaking down geographical and financial barriers. 
    These technological developments are not only enhancing productivity but also contributing to a more interconnected and informed society.
"""

template = """Summarize the {content} in one sentence.
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
summarize_chain = (
    prompt
    | llm 
    | StrOutputParser()
)

# Run the chain
summary = summarize_chain.invoke({"content": content})
print(summary)

### Question answering


Here is a Q&A agent built using the LCEL pattern.

This agent enables the LLM to learn from the provided content and answer questions based on what it has learned. Occasionally, if the LLM does not have sufficient information, it may generate a speculative answer. To manage this, we'll specifically instruct it to respond with "Unsure about the answer" if it is uncertain about the correct response.

The chain takes both the content (context) and question as inputs, processing them through our template before sending them to the LLM:


In [ ]:
content = """
    The solar system consists of the Sun, eight planets, their moons, dwarf planets, and smaller objects like asteroids and comets. 
    The inner planets—Mercury, Venus, Earth, and Mars—are rocky and solid. 
    The outer planets—Jupiter, Saturn, Uranus, and Neptune—are much larger and gaseous.
"""

question = "Which planets in the solar system are rocky and solid?"

template = """
    Answer the {question} based on the {content}.
    Respond "Unsure about answer" if not sure about the answer.
    
    Answer:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
qa_chain = (
    prompt
    | llm 
    | StrOutputParser()
)

# Run the chain
answer = qa_chain.invoke({"question": question, "content": content})
print(answer)

### Text classification


Here is a text classification agent designed to categorize text into predefined categories. This example employs zero-shot learning, where the agent classifies text without prior exposure to related examples.

Using the LCEL approach, we create a chain that takes both the text to be classified and the available categories as inputs:


In [ ]:
text = """
    The concert last night was an exhilarating experience with outstanding performances by all artists.
"""

categories = "Entertainment, Food and Dining, Technology, Literature, Music."

template = """
    Classify the {text} into one of the {categories}.
    
    Category:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
classification_chain = (
    prompt
    | llm 
    | StrOutputParser()
)

# Run the chain
category = classification_chain.invoke({"text": text, "categories": categories})
print(category)

### Code generation


Here is an example of an SQL code generation agent built with LCEL. This agent is designed to generate SQL queries based on provided descriptions. It interprets the requirements from your input and translates them into executable SQL code.

The chain takes your natural language description and transforms it into a properly formatted SQL query:


In [ ]:
description = """
    Retrieve the names and email addresses of all customers from the 'customers' table who have made a purchase in the last 30 days. 
    The table 'purchases' contains a column 'purchase_date'
"""

template = """
    Generate an SQL query based on the {description}
    
    SQL Query:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
sql_generation_chain = (
    prompt 
    | llm 
    | StrOutputParser()
)

# Run the chain
sql_query = sql_generation_chain.invoke({"description": description})
print(sql_query)

### Role playing


You can also configure the LLM to assume specific roles as defined by us, enabling it to follow predetermined rules and behave like a task-oriented chatbot.

This approach separates the role definition from the prompt structure, allowing for easy role-switching without rewriting the entire prompt. The key components are:

- `role`: Specifies the character, expertise, or persona the LLM should embody
- `tone`: Defines the communication style and emotional quality of responses
- `question`: Contains the user's query that needs addressing

By parameterizing these elements, you can rapidly change the LLM's behavior by adjusting single variables rather than rewriting entire prompts. This pattern is particularly valuable for building conversational agents that need to serve different functions or adapt to various contexts.

For example, the code below configures the LLM to act as a game master. In this role, the LLM answers questions about games while maintaining an engaging and immersive tone, enhancing the user experience. You can test the bot by asking questions related to tabletop role-playing games or game mastering. Try asking about game rules, storytelling techniques, player management, or setting descriptions such as:

1. "Who are you?"
2. "What are the basic rules of Dungeons & Dragons?"
3. "How do I create a balanced encounter for my players?"
4. "Can you describe a mysterious forest setting for my adventure?"
5. "What's a good puzzle I could use in my dungeon?"
6. "How do I handle a player who is constantly interrupting others?"

The function is written within a while loop, allowing continuous interaction. **To exit the loop and terminate the conversation, type "quit," "exit," or "bye" into the input box.**


In [ ]:
role = """
    Dungeon & Dragons game master
"""

tone = "engaging and immersive"

template = """
    You are an expert {role}. I have this question {question}. I would like our conversation to be {tone}.
    
    Answer:
    
"""
prompt = PromptTemplate.from_template(template)

# Create the LCEL chain
roleplay_chain = (
    prompt
    | llm 
    | StrOutputParser()
)

# Create an interactive chat loop
while True:
    query = input("Question: ")
    
    if query.lower() in ["quit", "exit", "bye"]:
        print("Answer: Goodbye!")
        break
        
    response = roleplay_chain.invoke({"role": role, "question": query, "tone": tone})
    print("Answer: ", response)

### Exercise 5

**Create an LCEL Chain with Custom Formatting**

In this exercise, you'll create your own LCEL chain that uses prompt templates to build a custom application.

**Task:** Create a product review analyzer that can:
1. Identify the sentiment (positive, negative, or neutral).
2. Extract mentioned product features.
3. Provide a one-sentence summary of the review.

**Steps:**
1. Create a prompt template with placeholders for the review text.
2. Build an LCEL chain that formats your prompt properly.
3. Process the sample reviews and display the results.
4. Try modifying the chain to change the output format.

**Sample input:**
```python
reviews = [
    "I love this smartphone! The camera quality is exceptional and the battery lasts all day. The only downside is that it heats up a bit during gaming.",
    "This laptop is terrible. It's slow, crashes frequently, and the keyboard stopped working after just two months. Customer service was unhelpful."
]


In [ ]:
## Starter code: provide your solutions in the TODO parts
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# First initialize your LLM
model_id = "gemini-3.8-flash" ## Or another free Gemini model, e.g. gemini-3.5-flash-lite or gemini-2.5-flash

# Use these parameters: temperature=0.2, max_output_tokens=1024

# TODO: Initialize your LLM with ChatGoogleGenerativeAI
llm = ## TODO

# Here is an example template you can use
template = """
Analyze the following product review:
"{review}"

Provide your analysis in the following format:
- Sentiment: (positive, negative, or neutral)
- Key Features Mentioned: (list the product features mentioned)
- Summary: (one-sentence summary)
"""

# TODO: Create your prompt template
product_review_prompt = ## TODO

# TODO: Build your LCEL chain (prompt | llm | output parser — no formatting function needed)
review_analysis_chain = (
    # Your code here
)

# Example reviews to process
reviews = [
    "I love this smartphone! The camera quality is exceptional and the battery lasts all day. The only downside is that it heats up a bit during gaming.",
    "This laptop is terrible. It's slow, crashes frequently, and the keyboard stopped working after just two months. Customer service was unhelpful."
]

# TODO: Process the reviews
for review in reviews:
    # Your code here
    pass

<details>
    <summary>Click here for hints</summary>

```python
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_google_genai import ChatGoogleGenerativeAI

model_id = "gemini-3.8-flash"

llm = ChatGoogleGenerativeAI(model=model_id, temperature=0.2, max_output_tokens=1024)

# Create the prompt template
template = """
Analyze the following product review:
"{review}"

Provide your analysis in the following format:
- Sentiment: (positive, negative, or neutral)
- Key Features Mentioned: (list the product features mentioned)
- Summary: (one-sentence summary)
"""

product_review_prompt = PromptTemplate.from_template(template)

# Build the LCEL chain
review_analysis_chain = (
    product_review_prompt
    | llm 
    | StrOutputParser()
)

# Process the reviews
reviews = [
    "I love this smartphone! The camera quality is exceptional and the battery lasts all day. The only downside is that it heats up a bit during gaming.",
    "This laptop is terrible. It's slow, crashes frequently, and the keyboard stopped working after just two months. Customer service was unhelpful."
]

for i, review in enumerate(reviews):
    print(f"==== Review #{i+1} ====")
    result = review_analysis_chain.invoke({"review": review})
    print(result)
    print()
```
</details>


In [ ]:
# @title ✅ Answer — Exercise 5 — LCEL product-review analyzer
run_answer = False  # @param {type:"boolean"}
# Try the exercise yourself first. Then tick run_answer (Colab) or set it to True, and run this cell.
# To read the code: click ▸ Show code (or double-click the cell title).
if not run_answer:
    class AnswerSkipped(Exception):
        def _render_traceback_(self): return ["⏸ Answer not run — tick run_answer above after you have tried the exercise."]
    raise AnswerSkipped()

# Exercise 5 — product review analyzer (LCEL)
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

model_id = "gemini-3.8-flash"
llm = ChatGoogleGenerativeAI(model=model_id, temperature=0.2, max_output_tokens=1024)

template = """
Analyze the following product review:
"{review}"

Provide your analysis in the following format:
- Sentiment: (positive, negative, or neutral)
- Key Features Mentioned: (list the product features mentioned)
- Summary: (one-sentence summary)
"""
product_review_prompt = PromptTemplate.from_template(template)

review_analysis_chain = product_review_prompt | llm | StrOutputParser()

reviews = [
    "I love this smartphone! The camera quality is exceptional and the battery lasts all day. The only downside is that it heats up a bit during gaming.",
    "This laptop is terrible. It's slow, crashes frequently, and the keyboard stopped working after just two months. Customer service was unhelpful."
]

for i, review in enumerate(reviews, start=1):
    print(f"==== Review #{i} ====")
    print(review_analysis_chain.invoke({"review": review}))
    print()

# Step 4 — change the output format: ask for JSON instead of bullets
json_prompt = PromptTemplate.from_template(
    'Analyze this product review: "{review}"\n'
    'Return JSON with keys "sentiment", "features" (list) and "summary".'
)
json_chain = json_prompt | llm | StrOutputParser()
print(json_chain.invoke({"review": reviews[0]}))

## 🏋️ Colab practice exercises

1. Run the same zero-shot, few-shot and CoT prompts with `gemini-3.8-flash`, `gemini-3.5-flash-lite` and `gemini-2.5-flash` (rebuild `llm` with another `model=`), and note quality and speed in a small table.
2. Change `temperature` (0.0 / 0.5 / 1.0) and `max_output_tokens` on the *'The wind is'* prompt; describe how output variety changes.
3. Self-consistency: call the age-puzzle prompt 5 times at `temperature=0.8` and take a majority vote in Python.
4. Rewrite Exercise 5 using `ChatPromptTemplate.from_messages` (system + human) piped into `llm`.
5. Build a `FewShotPromptTemplate` for emotion classification and compare it with the hand-written few-shot prompt.
6. Add `.with_structured_output()` (a Pydantic model with `sentiment`, `features`, `summary`) to the Exercise 5 chain and print the parsed objects.

In [ ]:
# Your exercise code here


In [ ]:
# @title ✅ Answer — 🏋️ Colab practice exercises 1–6
run_answer = False  # @param {type:"boolean"}
# Try the exercise yourself first. Then tick run_answer (Colab) or set it to True, and run this cell.
# To read the code: click ▸ Show code (or double-click the cell title).
if not run_answer:
    class AnswerSkipped(Exception):
        def _render_traceback_(self): return ["⏸ Answer not run — tick run_answer above after you have tried the exercise."]
    raise AnswerSkipped()

# 🏋️ Colab practice exercises — reference solutions
import time, re
from collections import Counter
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate, FewShotPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

# 1) Same prompts on three models — quality and speed table
test_prompts = {
    "zero-shot": "Classify as true or false: 'The Eiffel Tower is located in Berlin.' Answer:",
    "few-shot": ("Statement: 'I just won my first marathon!' Emotion: Joy\n"
                 "Statement: 'I lost my keys again.' Emotion: Frustration\n"
                 "Statement: 'That movie was so scary I had to cover my eyes.' Emotion:"),
    "cot": "A store had 22 apples, sold 15 and received 8. How many now? Think step by step.",
}
rows = []
for m in ["gemini-3.8-flash", "gemini-3.5-flash-lite", "gemini-2.5-flash"]:
    model = ChatGoogleGenerativeAI(model=m, temperature=0.2)
    for name, p in test_prompts.items():
        t0 = time.time()
        out = model.invoke(p).text
        rows.append((m, name, f"{time.time() - t0:.1f}s", out.strip().replace("\n", " ")[:60]))
print(f"{'model':24}{'prompt':11}{'time':7}answer")
for r in rows:
    print(f"{r[0]:24}{r[1]:11}{r[2]:7}{r[3]}")

# 2) temperature / max_output_tokens on "The wind is"
#    (per-call kwargs override the model settings; thinking_level="low" keeps thinking
#     from eating a small token budget, which would otherwise give an empty answer)
fast = ChatGoogleGenerativeAI(model="gemini-3.8-flash", thinking_level="low")
for t in [0.0, 0.5, 1.0]:
    for mt in [64, 256]:
        outs = {fast.invoke("The wind is ", temperature=t, max_output_tokens=mt).text.strip()[:70] for _ in range(2)}
        print(f"T={t} max={mt} distinct={len(outs)} -> {list(outs)[0]!r}")
# Expected: T=0.0 → (almost) identical runs; T=1.0 → different wording each run; small max → shorter/cut text.

# 3) Self-consistency with a majority vote
sc_llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.8)
q = ("When I was 6, my sister was half my age. Now I am 70, what age is my sister? "
     "Reason briefly, then end with 'Answer: <number>'.")
answers = []
for _ in range(5):
    txt = sc_llm.invoke(q).text
    nums = re.findall(r"Answer:\s*(\d+)", txt) or re.findall(r"\d+", txt)
    answers.append(nums[-1] if nums else None)
print("votes:", answers, "→ majority:", Counter(answers).most_common(1)[0][0])   # correct answer: 67

# 4) Exercise 5 with ChatPromptTemplate.from_messages (system + human)
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a product-review analyst. Reply with: Sentiment, Key Features Mentioned, Summary."),
    ("human", "Review: {review}"),
])
chat_chain = chat_prompt | llm | StrOutputParser()
print(chat_chain.invoke({"review": reviews[1]}))

# 5) FewShotPromptTemplate for emotion classification
examples = [
    {"statement": "I just won my first marathon!", "emotion": "Joy"},
    {"statement": "I can't believe I lost my keys again.", "emotion": "Frustration"},
    {"statement": "My best friend is moving to another country.", "emotion": "Sadness"},
]
example_prompt = PromptTemplate.from_template("Statement: {statement}\nEmotion: {emotion}")
few_shot = FewShotPromptTemplate(
    examples=examples, example_prompt=example_prompt,
    prefix="Classify the emotion of each statement with one word.",
    suffix="Statement: {input}\nEmotion:", input_variables=["input"],
)
print(few_shot.format(input="That movie was so scary I had to cover my eyes."))
print((few_shot | llm | StrOutputParser()).invoke({"input": "That movie was so scary I had to cover my eyes."}))  # → Fear

# 6) Structured output with Pydantic
class ReviewAnalysis(BaseModel):
    sentiment: str = Field(description="positive, negative or neutral")
    features: list[str] = Field(description="product features mentioned")
    summary: str = Field(description="one-sentence summary")

structured_chain = product_review_prompt | llm.with_structured_output(ReviewAnalysis)
for r in reviews:
    res = structured_chain.invoke({"review": r})
    print(type(res).__name__, res.sentiment, res.features, res.summary, sep=" | ")

## Conclusion

Congratulations on completing this lab on prompt engineering and LangChain prompt templates! You've successfully navigated from basic prompting techniques to more advanced approaches including zero-shot, one-shot, few-shot learning, as well as chain-of-thought reasoning and self-consistency prompting. You then applied these concepts practically using LangChain's prompt templates with the modern LCEL pattern to create various applications.

By learning how to properly structure prompts and build composable chains with the pipe operator (`|`), you've gained essential skills for developing robust LLM applications. These techniques provide a solid foundation for creating more complex systems and getting the most out of any language model you work with.


## Authors


[Hailey Quach](https://www.haileyq.com) is a Data Scientist at IBM.

[Kang Wang](https://author.skills.network/instructors/kang_wang) is a Data Scientist at IBM. He is also a PhD Candidate in the University of Waterloo.

[Faranak Heidari](https://author.skills.network/instructors/faranak_heidari) is a Data Scientist at IBM. 


<!---
## Change log

|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2025-03-07|1.1|Hailey Quach|Updated lab|
|2025-04-03|1.2|Jojy John|ID Reviewed|
|2025-04-03|1.2|Rahul Rawat|QA pass|
--->


© Copyright IBM Corporation. All rights reserved.
